In [1]:
import pandas as pd
import requests
import time
import json

In [2]:
books_df = pd.read_csv('books_clean.csv')  

In [3]:
# ======================
# 1. GET TOP 50 MOST FREQUENT AUTHOR NAMES
# ======================
print("GETTING TOP 50 MOST FREQUENT AUTHOR NAMES")
print("-" * 50)

# Count how many times each author name appears
name_counts = books_df['author_first_name'].value_counts().reset_index()
name_counts.columns = ['author_first_name', 'book_count']

# Get top 50 names
top_names = name_counts.head(50)

print(f"Top {len(top_names)} most frequent author names:")
print(top_names.to_string(index=False))

GETTING TOP 50 MOST FREQUENT AUTHOR NAMES
--------------------------------------------------
Top 50 most frequent author names:
author_first_name  book_count
             John        4897
           Robert        3281
            David        3250
          William        2647
            James        2511
          Michael        2175
          Richard        2044
             Paul        1529
            Peter        1513
          Charles        1506
           Thomas        1373
           George        1217
             Mark        1130
             Mary        1120
          Stephen        1014
           Joseph         908
           Edward         830
            Susan         822
        Elizabeth         745
          Barbara         738
             Jack         693
            Henry         666
             Alan         656
      Christopher         649
              Tom         632
            Frank         619
           Arthur         608
           Andrew         604
  

In [4]:
# ======================
# 2. GENDER INFERENCE FUNCTION
# ======================
def get_gender_from_name(name):
    """
    Get gender from name using genderize.io API
    Returns (gender, probability) or (None, 0) if failed
    """
    try:
        # Clean the name - just basic cleaning
        clean_name = str(name).strip().title()
        
        # Skip if too short
        if len(clean_name) < 2:
            return None, 0.0
        
        # Call the API
        url = f"https://api.genderize.io/?name={clean_name}"
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data.get('probability', 0) > 0.6:  # Accept if >60% confident
                return data.get('gender'), data.get('probability')
        
        return None, 0.0
        
    except Exception as e:
        print(f"  Error for {name}: {str(e)[:50]}")
        return None, 0.0

In [5]:
# ======================
# 3. GET GENDER FOR TOP NAMES
# ======================
print("2. GETTING GENDER FOR TOP NAMES")
print("-" * 50)

gender_results = []

for idx, row in top_names.iterrows():
    name = row['author_first_name']
    book_count = row['book_count']
    
    print(f"  Processing {idx+1}/{len(top_names)}: {name}...")
    
    # Get gender from API
    gender, probability = get_gender_from_name(name)
    
    # Add to results
    gender_results.append({
        'author_first_name': name,
        'book_count': book_count,
        'inferred_gender': gender,
        'gender_confidence': probability
    })
    
    # Rate limiting - wait 0.5 seconds between requests
    time.sleep(0.5)

2. GETTING GENDER FOR TOP NAMES
--------------------------------------------------
  Processing 1/50: John...
  Processing 2/50: Robert...
  Processing 3/50: David...
  Processing 4/50: William...
  Processing 5/50: James...
  Processing 6/50: Michael...
  Processing 7/50: Richard...
  Processing 8/50: Paul...
  Processing 9/50: Peter...
  Processing 10/50: Charles...
  Processing 11/50: Thomas...
  Processing 12/50: George...
  Processing 13/50: Mark...
  Processing 14/50: Mary...
  Processing 15/50: Stephen...
  Processing 16/50: Joseph...
  Processing 17/50: Edward...
  Processing 18/50: Susan...
  Processing 19/50: Elizabeth...
  Processing 20/50: Barbara...
  Processing 21/50: Jack...
  Processing 22/50: Henry...
  Processing 23/50: Alan...
  Processing 24/50: Christopher...
  Processing 25/50: Tom...
  Processing 26/50: Frank...
  Processing 27/50: Arthur...
  Processing 28/50: Andrew...
  Processing 29/50: Philip...
  Processing 30/50: Daniel...
  Processing 31/50: Jane...
  Pro

In [6]:
# Create DataFrame
gender_df = pd.DataFrame(gender_results)

In [7]:
gender_df

,author_first_name,book_count,inferred_gender,gender_confidence
0,John,4897,male,1.00
1,Robert,3281,male,1.00
2,David,3250,male,1.00
3,William,2647,male,1.00
4,James,2511,male,1.00
5,Michael,2175,male,1.00
6,Richard,2044,male,1.00
7,Paul,1529,male,1.00
8,Peter,1513,male,1.00
9,Charles,1506,male,1.00


In [8]:
# Save to CSV
output_filename = 'author_genders.csv'
gender_df.to_csv(output_filename, index=False)